# 🧠 Entrenamiento de Modelos IA para Clasificación de Tickets
**Proyecto Soporte Tickets**

¡Bienvenido al motor de Inteligencia Artificial de nuestra plataforma! Este notebook documenta paso a paso el proceso de entrenamiento de los modelos de Machine Learning encargados de clasificar automáticamente la **Categoría** y la **Prioridad** de los tickets de soporte entrantes.

Hemos estructurado este análisis utilizando la metodología **CRISP-DM** para garantizar un flujo de trabajo ordenado, profesional e **iterativo**.

---

## 1. Comprensión del Negocio (Business Understanding)
El objetivo principal de este módulo es reducir el tiempo de triaje (clasificación manual) en nuestra mesa de ayuda. Nuestro sistema web en Django recibe tickets de usuarios, y para agilizar la respuesta, la IA debe predecir a qué categoría pertenece el problema y qué nivel de urgencia tiene, delegando automáticamente el ticket al técnico correspondiente.

## 2. Preparación del Entorno
Comenzamos importando las librerías necesarias para la manipulación de datos (`pandas`), las herramientas de Machine Learning (`scikit-learn`) y la persistencia de los modelos (`joblib`).

*Nota de arquitectura:* Este script se ejecuta desde la carpeta `machine_learning`, por lo que las rutas dinámicas (`BASE_DIR`) buscarán los datos en el directorio local `dataset/` y exportarán los modelos listos para producción a la carpeta `modelos/`, donde posteriormente serán consumidos por nuestro backend en Django.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import joblib
import os

# 1. CONFIGURACIÓN DE RUTAS

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

DATASET_PATH = os.path.join(
    BASE_DIR,
    'dataset',
    'tickets.csv'
)

MODELOS_DIR = os.path.join(
    BASE_DIR,
    'modelos'
)

os.makedirs(
    MODELOS_DIR,
    exist_ok=True
)

## 3. Comprensión de los Datos (Data Understanding)
Antes de enseñar a la máquina, necesitamos saber con qué estamos trabajando. En esta fase cargamos nuestro historial de tickets (el archivo `tickets.csv`) y exploramos su estructura base: cuántos registros tenemos y qué columnas conforman nuestro conocimiento previo.

In [ ]:
# 2. CARGAR DATASET
try:
    df = pd.read_csv(DATASET_PATH)
    print("\n" + "=" * 60)
    print("DATASET CARGADO")
    print("=" * 60)
    print(df.head())
    print("\nColumnas del dataset:")
    print(df.columns.tolist())
    print("\nCantidad de registros:")
    print(len(df))
except FileNotFoundError:
    print(f"Archivo no encontrado en la ruta: {DATASET_PATH}. Asegúrate de tener el dataset.")
    # Creando un dataset dummy para que el notebook corra como ejemplo si no existe
    df = pd.DataFrame({
        'asunto': ['Error red', 'Falla mouse', 'Pantalla azul', 'Olvido clave'],
        'descripcion': ['No hay internet', 'Click no funciona', 'Windows muere', 'No recuerdo password'],
        'categoria': ['Redes', 'Hardware', 'Sistema', 'Accesos'],
        'prioridad': ['Alta', 'Baja', 'Critica', 'Media']
    })

## 4. Preparación de los Datos (Iteración 1 - Baseline)
Los modelos de Machine Learning y el texto vacío (valores nulos o `NaN`) no se llevan bien. En esta etapa de limpieza, rellenamos cualquier espacio vacío.

En esta **primera iteración** de CRISP-DM, crearemos un modelo base (baseline) utilizando **únicamente el campo `asunto`** para predecir, con el objetivo de establecer un punto de partida para nuestras métricas.

In [ ]:
# 3. LIMPIAR DATOS
df['asunto'] = df['asunto'].fillna('')
df['descripcion'] = df['descripcion'].fillna('')
df['categoria'] = df['categoria'].fillna('')
df['prioridad'] = df['prioridad'].fillna('')

# ITERACIÓN 1: Usar solo el asunto como característica predictiva
df['texto_iteracion_1'] = df['asunto']

## 5. Modelado y Evaluación (Iteración 1 - Baseline)
Aquí es donde ocurre la magia. Hemos encapsulado el proceso de entrenamiento en una función robusta `entrenar_objetivo`. 
Primero, evaluamos nuestro modelo base utilizando únicamente el asunto del ticket para predecir la Categoría.

In [ ]:
# FUNCIÓN PARA ENTRENAR UN OBJETIVO
def entrenar_objetivo(X, y, nombre_objetivo):
    # (NOTA: Si el dataset es muy pequeño, stratify fallará, ajustamos dinámicamente para el ejemplo)
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    print("\n" + "=" * 60)
    print(f"ENTRENANDO: {nombre_objetivo.upper()}")
    print("=" * 60)

    vectorizador = TfidfVectorizer(lowercase=True, stop_words=None)
    X_train_tfidf = vectorizador.fit_transform(X_train)
    X_test_tfidf = vectorizador.transform(X_test)

    def evaluar_modelo(nombre, modelo):
        modelo.fit(X_train_tfidf, y_train)
        predicciones = modelo.predict(X_test_tfidf)
        accuracy = accuracy_score(y_test, predicciones)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, predicciones, average='macro', zero_division=0)
        
        print("\n" + "-" * 50)
        print(nombre)
        print("-" * 50)
        print(f"F1 Macro: {f1:.4f} | Accuracy: {accuracy:.4f}")
        return modelo, f1

    modelo_nb, f1_nb = evaluar_modelo("NAIVE BAYES", MultinomialNB())
    modelo_lr, f1_lr = evaluar_modelo("LOGISTIC REGRESSION", LogisticRegression(max_iter=1000))
    modelo_svm, f1_svm = evaluar_modelo("SVM", SVC(kernel='linear', probability=True))

    modelos = {'naive_bayes': (modelo_nb, f1_nb), 'logistic_regression': (modelo_lr, f1_lr), 'svm': (modelo_svm, f1_svm)}
    mejor_nombre = max(modelos, key=lambda nombre: modelos[nombre][1])
    mejor_modelo = modelos[mejor_nombre][0]
    mejor_f1 = modelos[mejor_nombre][1]

    print("\n" + "=" * 60)
    print(f"MEJOR MODELO PARA {nombre_objetivo.upper()}: {mejor_nombre} (F1: {mejor_f1:.4f})")
    print("=" * 60)

    # GUARDAR MODELO
    paquete_modelo = {'modelo': mejor_modelo, 'vectorizador': vectorizador, 'nombre_modelo': mejor_nombre, 'f1_macro': mejor_f1}
    nombre_archivo = f'modelo_{nombre_objetivo}.joblib'
    ruta_modelo = os.path.join(MODELOS_DIR, nombre_archivo)
    joblib.dump(paquete_modelo, ruta_modelo)
    print(f"Modelo guardado: {ruta_modelo}")

print("--- RESULTADOS ITERACIÓN 1 (SOLO ASUNTO) ---")
entrenar_objetivo(df['texto_iteracion_1'], df['categoria'], 'categoria_iteracion_1')

## 🔄 Ciclo CRISP-DM: Análisis y Nueva Iteración (Iteración 2)
**Evaluación de la Iteración 1:** Notamos que usar solo el "asunto" limita la capacidad del modelo para clasificar correctamente tickets ambiguos, ya que falta contexto semántico.

**Decisión (Retorno a Preparación de Datos):** Siguiendo fielmente la metodología iterativa CRISP-DM, volvemos a la Fase 4. Aplicaremos *Feature Engineering* para fusionar el `asunto` y la `descripcion` en una sola variable (`texto`), aportando mayor densidad de información al modelo, y volveremos a entrenar y evaluar.

In [ ]:
# NUEVA PREPARACIÓN DE DATOS (Iteración 2)
# Unimos asunto y descripción para dar más contexto
df['texto_iteracion_2'] = df['asunto'] + ' ' + df['descripcion']

print("\n\n--- RESULTADOS ITERACIÓN 2 (ASUNTO + DESCRIPCIÓN) ---")
# NUEVO MODELADO Y EVALUACIÓN (Iteración 2 - Modelos Finales)
entrenar_objetivo(df['texto_iteracion_2'], df['categoria'], 'categoria_final')
entrenar_objetivo(df['texto_iteracion_2'], df['prioridad'], 'prioridad_final')

## 6. Despliegue (Deployment)
Tras nuestra segunda iteración del ciclo CRISP-DM, hemos logrado modelos más robustos. 
Los modelos ganadores finales de la Iteración 2 (empaquetados junto a sus respectivos vectorizadores TF-IDF) han sido exportados automáticamente mediante `.joblib` hacia la carpeta `modelos/`, quedando listos para ser integrados al backend en Django.

In [ ]:
print("\n" + "=" * 60)
print("ENTRENAMIENTO Y CICLO CRISP-DM COMPLETADOS")
print("=" * 60)
print("\nModelos de producción listos en:")
print(os.path.join(MODELOS_DIR, 'modelo_categoria_final.joblib'))
print(os.path.join(MODELOS_DIR, 'modelo_prioridad_final.joblib'))